In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

file_path = Path('../data/raw/data.csv')
df = pd.read_csv(file_path, sep=';')

# Re-apply fixes from Notebook 1
df.columns = df.columns.str.strip()
df['Target'] = df['Target'].map({'Dropout': 1, 'Graduate': 0, 'Enrolled': 0})

print(f"Shape: {df.shape}")
print(df['Target'].value_counts())

Shape: (4424, 37)
Target
0    3003
1    1421
Name: count, dtype: int64


In [6]:
# Determine academic performance ratios
df['success_rate_1st_sem'] = df['Curricular units 1st sem (approved)'] / \
    df['Curricular units 1st sem (enrolled)'].replace(0, np.nan)

df['success_rate_2nd_sem'] = df['Curricular units 2nd sem (approved)'] / \
    df['Curricular units 2nd sem (enrolled)'].replace(0, np.nan)

# Determine overall academic performance across both semesters
df['overall_approval_rate'] = (
    df['Curricular units 1st sem (approved)'] + df['Curricular units 2nd sem (approved)']
) / (
    df['Curricular units 1st sem (enrolled)'] + df['Curricular units 2nd sem (enrolled)']
).replace(0, np.nan)

# Determine the average grade across both semesters
df['average_grade'] = (
    df['Curricular units 1st sem (grade)'] + df['Curricular units 2nd sem (grade)']
) / 2

# Create a financial risk flag
df['financial_risk'] = ((df['Debtor'] == 1) | (df['Tuition fees up to date'] == 0)).astype(int)

# Fill any NaN create by division by zero
df.fillna(0, inplace=True)

print("New features added:")
print(df[['success_rate_1st_sem', 'success_rate_2nd_sem', 
          'overall_approval_rate', 'average_grade', 'financial_risk']].head())


New features added:
   success_rate_1st_sem  success_rate_2nd_sem  overall_approval_rate  \
0              0.000000              0.000000               0.000000   
1              1.000000              1.000000               1.000000   
2              0.000000              0.000000               0.000000   
3              1.000000              0.833333               0.916667   
4              0.833333              1.000000               0.916667   

   average_grade  financial_risk  
0       0.000000               0  
1      13.833333               1  
2       0.000000               1  
3      12.914286               0  
4      12.666667               0  


In [9]:
# Split the data
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Target'])
y = df['Target']

# Split the training and test data into an 80/20 fold
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]} rows")
print(f"Test size: {X_test.shape[0]} rows")

Train size: 3539 rows
Test size: 885 rows


In [11]:
processed_path = Path('../data/processed')

X_train.to_csv(processed_path / 'X_train.csv', index=False)
X_test.to_csv(processed_path / 'X_test.csv', index=False)
y_train.to_csv(processed_path / 'y_train.csv', index=False)
y_test.to_csv(processed_path / 'y_test.csv', index=False)

print("All splits saved to data/processed/")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

✅ All splits saved to data/processed/
X_train: (3539, 41)
X_test:  (885, 41)
